In [1]:
import os
import cv2
import json
import glob

from ultralytics import YOLO
from tqdm import tqdm
from collections import Counter

In [2]:
DATA_ROOT = r"Sample (1)\Sample"

VIDEO_ROOT = os.path.join(
    DATA_ROOT,
    "01.원천데이터"
)

OUTPUT_DIR = "B_PART_LIGHT"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [11]:
model = YOLO("yolov8s.pt")

FRAME_INTERVAL = 10

CONF_THRESHOLD = 0.15

TARGET_CLASSES = {
    "person",
    "bicycle",
    "motorcycle"
}

In [12]:
target_videos = []

keywords = [
    "이륜 이동수단 인도 주행",
    "이륜 이동수단 헬멧 미착용"
]

for keyword in keywords:

    paths = glob.glob(

        os.path.join(
            VIDEO_ROOT,
            keyword,
            "**",
            "*.mp4"
        ),

        recursive=True
    )

    target_videos.extend(paths)

print("videos:", len(target_videos))

for v in target_videos:
    print(os.path.basename(v))

videos: 10
ms_e3500_c1.mp4
ms_e3500_c2.mp4
bs_e4387_c1.mp4
bs_e4387_c2.mp4
es_e3768_c1.mp4
es_e3768_c2.mp4
md_e2122_c1.mp4
md_e2122_c2.mp4
mp_e2730_c1.mp4
mp_e2730_c2.mp4


In [13]:
def detect_video(video_path):

    cap = cv2.VideoCapture(video_path)

    summary_counter = Counter()

    frame_counter = Counter()

    frames = []

    frame_id = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        if frame_id % FRAME_INTERVAL != 0:

            frame_id += 1
            continue

        result = model(
            frame,
            verbose=False,
            classes=[0,1,3],
            imgsz=1280
        )[0]

        objects = []

        frame_classes = set()

        for box in result.boxes:

            conf = float(box.conf[0])

            if conf < CONF_THRESHOLD:
                continue

            cls_id = int(box.cls[0])

            label = model.names[cls_id]

            if label not in TARGET_CLASSES:
                continue

            x1,y1,x2,y2 = map(
                int,
                box.xyxy[0]
            )

            objects.append({

                "class": label,

                "confidence": round(
                    conf,
                    4
                ),

                "bbox":[
                    x1,y1,x2,y2
                ]
            })

            summary_counter[label] += 1

            frame_classes.add(label)

        for c in frame_classes:

            frame_counter[c] += 1

        frames.append({

            "frame_id": frame_id,

            "classes": list(frame_classes),

            "object_count": len(objects),

            "objects": objects
        })

        frame_id += 1

    cap.release()

    return {

        "summary": dict(summary_counter),

        "frame_summary": dict(frame_counter),

        "frames": frames
    }

In [14]:
for video_path in tqdm(target_videos):

    video_name = os.path.basename(
        video_path
    )

    print(
        f"\nProcessing {video_name}"
    )

    result = detect_video(
        video_path
    )

    save_path = os.path.join(

        OUTPUT_DIR,

        video_name.replace(
            ".mp4",
            ".json"
        )
    )

    with open(
        save_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=4
        )

  0%|                                                                                           | 0/10 [00:00<?, ?it/s]


Processing ms_e3500_c1.mp4


 10%|████████▎                                                                          | 1/10 [00:31<04:47, 31.97s/it]


Processing ms_e3500_c2.mp4


 20%|████████████████▌                                                                  | 2/10 [01:01<04:06, 30.75s/it]

...

 80%|██████████████████████████████████████████████████████████████████▍                | 8/10 [04:03<00:59, 30.00s/it]


Processing mp_e2730_c1.mp4


 90%|██████████████████████████████████████████████████████████████████████████▋        | 9/10 [04:33<00:29, 29.96s/it]


Processing mp_e2730_c2.mp4


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [05:02<00:00, 30.26s/it]


In [15]:
global_summary = Counter()

global_frame_summary = Counter()

json_files = glob.glob(
    os.path.join(
        OUTPUT_DIR,
        "*.json"
    )
)

for path in json_files:

    with open(
        path,
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    global_summary.update(
        data["summary"]
    )

    global_frame_summary.update(
        data["frame_summary"]
    )

print("\n=== OBJECT SUMMARY ===")
print(global_summary)

print("\n=== FRAME SUMMARY ===")
print(global_frame_summary)


=== OBJECT SUMMARY ===
Counter({'person': 909, 'motorcycle': 216, 'bicycle': 41})

=== FRAME SUMMARY ===
Counter({'person': 404, 'motorcycle': 169, 'bicycle': 31})


In [16]:
sample_json = json_files[0]

with open(
    sample_json,
    encoding="utf-8"
) as f:

    data = json.load(f)

print(
    json.dumps(
        data["summary"],
        ensure_ascii=False,
        indent=4
    )
)

print()

print(
    json.dumps(
        data["frame_summary"],
        ensure_ascii=False,
        indent=4
    )
)

{
    "person": 34,
    "bicycle": 24
}

{
    "person": 30,
    "bicycle": 18
}


In [17]:
for path in json_files:

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    print("\n")
    print(os.path.basename(path))

    print(
        data["summary"]
    )



bs_e4387_c1.json
{'person': 34, 'bicycle': 24}


bs_e4387_c2.json
{'person': 30, 'bicycle': 7}


es_e3768_c1.json
{'person': 38, 'bicycle': 2, 'motorcycle': 19}


es_e3768_c2.json
{'person': 240, 'bicycle': 8, 'motorcycle': 25}


md_e2122_c1.json
{'person': 348, 'motorcycle': 91}


md_e2122_c2.json
{'person': 132, 'motorcycle': 24}


mp_e2730_c1.json
{'person': 40, 'motorcycle': 31}


mp_e2730_c2.json
{'person': 37, 'motorcycle': 26}


ms_e3500_c1.json
{'person': 10}


ms_e3500_c2.json
{}


In [18]:
#실패 셀 / 오토바이 인도 주행 에대한 셀 

video_path = [v for v in target_videos if "ms_e3500_c1" in v][0]

cap = cv2.VideoCapture(video_path)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(total_frames)
print(fps)

for idx in [0, total_frames//4, total_frames//2, total_frames*3//4]:

    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

    ret, frame = cap.read()

    if ret:
        cv2.imwrite(
            f"check_{idx}.jpg",
            frame
        )

cap.release()

957
30.0
